# 01 — Document Exploration

List raw documents and inspect extraction output. Paths use project config (CWD-independent).

**Prerequisites:** Run extraction first: `python -m src.ingestion.run_extraction`

**What you get:** Document list, extraction table, and per-document stats (character count, pages).

In [ ]:
from pathlib import Path
import pandas as pd

from src.config import RAW_DOCS_DIR
from src.ingestion.load_documents import list_document_paths

paths = list_document_paths(RAW_DOCS_DIR)
print(f"Found {len(paths)} documents")
for p in paths:
    print(f"  - {p.name}")

In [ ]:
from src.config import EXTRACTION_OUTPUT

if not EXTRACTION_OUTPUT.exists():
    raise FileNotFoundError("Run: python -m src.ingestion.run_extraction")
df = pd.read_csv(EXTRACTION_OUTPUT)
print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head(10)

In [ ]:
# Per-document stats: character count and page count
if "char_count" in df.columns:
    stats = df.groupby("document_name").agg(
        pages=("page_number", "count"),
        total_chars=("char_count", "sum"),
    ).reset_index()
    print("Per-document summary:")
    display(stats)
else:
    print("Columns:", list(df.columns))

In [ ]:
# Sample text from first document (first 400 chars)
if "text" in df.columns and len(df) > 0:
    sample = df.iloc[0]
    print(f"Document: {sample.get('document_name', '')} (page {sample.get('page_number', '')})")
    print("-" * 40)
    print((sample["text"] or "")[:400])
    print("...")